# SEC/CCM Unified Runner

This notebook bootstraps the canonical script implementation in `src/thesis_pkg/notebooks_and_scripts/sec_ccm_unified_runner.py`, but the editable Colab-facing configuration now lives directly in the notebook.

Run the cells top to bottom. In Colab, edit only the configuration cell before starting the final run cell.


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_ENV_VAR = "NLP_THESIS_REPO_ROOT"
GIT_URL_ENV_VAR = "NLP_THESIS_GIT_URL"
DEFAULT_GIT_URL = "https://github.com/Erew42/NLP_Thesis.git"
CLONE_ROOT = Path("/content/NLP_Thesis")


def _find_repo_root() -> Path | None:
    candidates: list[Path] = []
    env_root = os.environ.get(REPO_ENV_VAR)
    if env_root:
        candidates.append(Path(env_root).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    if IN_COLAB:
        candidates.extend(
            [
                CLONE_ROOT,
                Path("/content/drive/MyDrive/NLP_Thesis"),
                Path("/content/drive/My Drive/NLP_Thesis"),
            ]
        )

    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "src" / "thesis_pkg" / "pipeline.py").exists():
            return candidate
    return None


if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

repo_root = _find_repo_root()
if repo_root is None and IN_COLAB:
    git_url = os.environ.get(GIT_URL_ENV_VAR, DEFAULT_GIT_URL)
    if CLONE_ROOT.exists() and not (CLONE_ROOT / "src" / "thesis_pkg" / "pipeline.py").exists():
        raise FileExistsError(
            f"{CLONE_ROOT} exists but does not look like the NLP_Thesis repo. Remove it or set {REPO_ENV_VAR}."
        )
    if not CLONE_ROOT.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", git_url, str(CLONE_ROOT)])
    repo_root = CLONE_ROOT

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate the NLP_Thesis checkout. Run this notebook from the repo root, set NLP_THESIS_REPO_ROOT, or use Colab so the notebook can clone the repo automatically."
    )

os.environ[REPO_ENV_VAR] = str(repo_root)
src = repo_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

if IN_COLAB and os.environ.get("SEC_CCM_SKIP_PIP_INSTALL", "0") != "1":
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "--editable", str(repo_root)]
    )

print({"IN_COLAB": IN_COLAB, "repo_root": str(repo_root), "src_exists": src.exists()})


In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path


def _resolve_colab_drive_root() -> Path:
    for candidate in (
        Path("/content/drive/MyDrive"),
        Path("/content/drive/My Drive"),
        Path("/content/drive"),
    ):
        if candidate.exists():
            return candidate
    return Path("/content/drive")


DATA_PROFILE = "DRIVE_FULL" if IN_COLAB else "LOCAL_SAMPLE"
SAMPLE_ROOT = repo_root / "full_data_run" / "sample_5pct_seed42"
WORK_ROOT = (
    _resolve_colab_drive_root() / "Data_LM"
    if IN_COLAB
    else Path("C:/Users/erik9/Documents/SEC_Data")
)

if DATA_PROFILE == "LOCAL_SAMPLE":
    SEC_ZIP_DIR = SAMPLE_ROOT / "10_X_reports"
    SEC_BATCH_ROOT = SAMPLE_ROOT / "sec_batches"
    SEC_YEAR_MERGED_DIR = SAMPLE_ROOT / "year_merged"
    SEC_LIGHT_METADATA_PATH = SAMPLE_ROOT / "derived_data" / "filings_metadata_LIGHT.sample_5pct_seed42.parquet"
    CCM_BASE_DIR = SAMPLE_ROOT / "ccm_parquet_data"
    CCM_DERIVED_DIR = SAMPLE_ROOT / "derived_data"
    CCM_REUSE_DAILY_PATH = SAMPLE_ROOT / "derived_data" / "final_flagged_data_compdesc_added.sample_5pct_seed42.parquet"
    CANONICAL_LINK_NAME = "canonical_link_table_after_startdate_change.sample_5pct_seed42.parquet"
    CCM_DAILY_NAME = "final_flagged_data_compdesc_added.sample_5pct_seed42.parquet"
    RUN_ROOT = SAMPLE_ROOT / "results" / "sec_ccm_unified_runner" / "local_sample"
    RUN_SEC_PARSE = False  # True | False | "auto"
    RUN_SEC_YEARLY_MERGE = False  # True | False | "auto"
else:
    SEC_ZIP_DIR = WORK_ROOT / "Zip_Files"
    SEC_BATCH_ROOT = WORK_ROOT / "parquet_data"
    SEC_YEAR_MERGED_DIR = SEC_BATCH_ROOT / "_year_merged"
    SEC_LIGHT_METADATA_PATH = SEC_BATCH_ROOT / "filings_metadata_1993_2024_LIGHT.parquet"
    CCM_BASE_DIR = WORK_ROOT / "CRSP_Compustat_data" / "parquet_data"
    CCM_DERIVED_DIR = WORK_ROOT / "CRSP_Compustat_data" / "derived_data"
    CCM_REUSE_DAILY_PATH = CCM_DERIVED_DIR / "final_flagged_data_compdesc_added.parquet"
    CANONICAL_LINK_NAME = "canonical_link_table_after_startdate_change.parquet"
    CCM_DAILY_NAME = "final_flagged_data_compdesc_added.parquet"
    RUN_ROOT = WORK_ROOT / "results" / "sec_ccm_unified_runner"
    RUN_SEC_PARSE = False  # True | False | "auto"
    RUN_SEC_YEARLY_MERGE = False  # True | False | "auto"

# Editable folder specification
SEC_CCM_OUTPUT_DIR = RUN_ROOT / "sec_ccm_premerge"
SEC_ITEMS_ANALYSIS_DIR = RUN_ROOT / "items_analysis"
SEC_ITEMS_DIAGNOSTIC_DIR = RUN_ROOT / "items_diagnostic"
SEC_NO_ITEM_DIR = RUN_ROOT / "no_item_diagnostics"
BOUNDARY_OUT_DIR = RUN_ROOT / "boundary_diagnostics"
BOUNDARY_INPUT_DIR = BOUNDARY_OUT_DIR / "matched_filings_input"
REFINITIV_STEP1_OUT_DIR = RUN_ROOT / "refinitiv_step1"
REFINITIV_OWNERSHIP_UNIVERSE_DIR = REFINITIV_STEP1_OUT_DIR / "ownership_universe_common_stock"
REFINITIV_OWNERSHIP_AUTHORITY_DIR = REFINITIV_STEP1_OUT_DIR / "ownership_authority_common_stock"
REFINITIV_DOC_OWNERSHIP_LM2011_DIR = RUN_ROOT / "refinitiv_doc_ownership_lm2011"
LM2011_ADDITIONAL_DATA_DIR = repo_root / "full_data_run" / "LM2011_additional_data"
LM2011_FF48_SICCODES_PATH = LM2011_ADDITIONAL_DATA_DIR / "FF_Siccodes_48_Industries.txt"
LM2011_OUTPUT_DIR = RUN_ROOT / "lm2011_post_refinitiv"
FINBERT_OUTPUT_DIR = RUN_ROOT / "finbert_item_analysis"

LOCAL_TMP = Path("/content/_tmp_zip") if IN_COLAB else repo_root / ".tmp" / "zip"
LOCAL_WORK = Path("/content/_batch_work") if IN_COLAB else repo_root / ".tmp" / "batch_work"
LOCAL_ITEM_WORK = Path("/content/_item_work") if IN_COLAB else repo_root / ".tmp" / "item_work"
LOCAL_MERGE_WORK = Path("/content/_merge_work") if IN_COLAB else repo_root / ".tmp" / "merge_work"

# Run toggles
RUN_CCM_MODE = "REUSE"  # REBUILD | REUSE
RUN_SEC_CCM_PREMERGE = False
RUN_REFINITIV_STEP1 = True
RUN_REFINITIV_STEP1_RESOLUTION = True
RUN_REFINITIV_OWNERSHIP_UNIVERSE_HANDOFF = True
RUN_REFINITIV_OWNERSHIP_UNIVERSE_RESULTS = True
RUN_REFINITIV_OWNERSHIP_AUTHORITY = True
RUN_REFINITIV_DOC_OWNERSHIP_LM2011_EXACT_HANDOFF = True
RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FALLBACK_HANDOFF = True
RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FINALIZE = True
RUN_REFINITIV_DOC_ANALYST_LM2011_ANCHORS = True
RUN_REFINITIV_DOC_ANALYST_LM2011_SELECT = True
RUN_REFINITIV_INSTRUMENT_AUTHORITY = True
RUN_REFINITIV_ANALYST_REQUEST_GROUPS = True
RUN_REFINITIV_ANALYST_ACTUALS = True
RUN_REFINITIV_ANALYST_ESTIMATES_MONTHLY = True
RUN_REFINITIV_ANALYST_NORMALIZE = True
RUN_GATED_ITEM_EXTRACTION = False
RUN_UNMATCHED_DIAGNOSTIC_TRACK = False
RUN_NO_ITEM_DIAGNOSTICS = False
RUN_BOUNDARY_DIAGNOSTICS = False
RUN_VALIDATION_CHECKS = False

# Downstream toggles
RUN_LM2011_POST_REFINITIV = False
RUN_LM2011_SAMPLE_BACKBONE = None  # True | False | None -> inherit umbrella default
RUN_LM2011_ANNUAL_ACCOUNTING_PANEL = None
RUN_LM2011_QUARTERLY_ACCOUNTING_PANEL = None
RUN_LM2011_FF_FACTORS_DAILY_NORMALIZED = None
RUN_LM2011_FF_FACTORS_MONTHLY_WITH_MOM_NORMALIZED = None
RUN_LM2011_TEXT_FEATURES_FULL_10K = None
RUN_LM2011_TEXT_FEATURES_MDA = None
RUN_LM2011_EVENT_SCREEN_SURFACE = None
RUN_LM2011_TABLE_I_SAMPLE_CREATION = None
RUN_LM2011_TABLE_I_SAMPLE_CREATION_1994_2024 = None
RUN_LM2011_EVENT_PANEL = None
RUN_LM2011_SUE_PANEL = None
RUN_LM2011_RETURN_REGRESSION_PANEL_FULL_10K = None
RUN_LM2011_RETURN_REGRESSION_PANEL_MDA = None
RUN_LM2011_SUE_REGRESSION_PANEL = None
RUN_LM2011_TABLE_IV_RESULTS = None
RUN_LM2011_TABLE_V_RESULTS = None
RUN_LM2011_TABLE_VI_RESULTS = None
RUN_LM2011_TABLE_VIII_RESULTS = None
RUN_LM2011_TABLE_IA_I_RESULTS = None
RUN_LM2011_TRADING_STRATEGY_MONTHLY_RETURNS = None
RUN_LM2011_TABLE_IA_II_RESULTS = None

RUN_FINBERT = False
RUN_FINBERT_PREPROCESS = False
RUN_FINBERT_ANALYSIS = False

# Other configuration
SEC_PARSE_MODE = "parsed"  # raw | parsed
YEARS = list(range(1993, 2025))
ITEM_EXTRACTION_REGIME = "legacy"
FORMS_10K_10Q = [
    "10-K",
    "10-K/A",
    "10-KA",
    "10-Q",
    "10-Q/A",
    "10-QA",
    "10-KT",
    "10-KT/A",
    "10-QT",
    "10-QT/A",
    "10-K405",
]
DAILY_FEATURE_COLUMNS = [
    "RET",
    "RETX",
    "PRC",
    "FINAL_PRC",
    "BIDLO",
    "ASKHI",
    "VOL",
    "TCAP",
    "SHROUT",
    "TICKER",
    "SHRCD",
    "EXCHCD",
]
REQUIRED_DAILY_NON_NULL_FEATURES = ["RET"]
LSEG_REQUEST_MIN_DATE = "1994-01-01"  # ISO date | None
LSEG_REQUEST_MAX_DATE = "2024-12-31"  # ISO date | None

# Downstream LM2011 parameters
LM2011_SAMPLE_BACKBONE_PATH = None  # None -> use script resolution
LM2011_MATCHED_CLEAN_PATH = None
LM2011_DAILY_PANEL_PATH = None
LM2011_ITEMS_ANALYSIS_DIR = None
LM2011_CCM_BASE_DIR = None
LM2011_YEAR_MERGED_DIR = None
LM2011_MONTHLY_STOCK_PATH = None
LM2011_FF_MONTHLY_WITH_MOM_PATH = None
LM2011_FULL_10K_CLEANING_CONTRACT = "current"
LM2011_TEXT_FEATURE_BATCH_SIZE = 1000
LM2011_EVENT_WINDOW_DOC_BATCH_SIZE = 250

# Downstream FinBERT parameters
FINBERT_SOURCE_ITEMS_DIR = None  # None -> use script resolution
FINBERT_BACKBONE_PATH = None
FINBERT_YEARS = None
FINBERT_RUN_NAME = None
FINBERT_DEVICE = None
FINBERT_WRITE_SENTENCE_SCORES = False
FINBERT_SENTENCE_POSTPROCESS_POLICY = "reference_stitch_protect_v3"
FINBERT_OVERWRITE = False
FINBERT_NOTE = ""
FINBERT_BATCH_PROFILE = "baseline"
FINBERT_SHORT_BATCH_SIZE = None
FINBERT_MEDIUM_BATCH_SIZE = None
FINBERT_LONG_BATCH_SIZE = None
FINBERT_SHORT_MAX_LENGTH = None
FINBERT_MEDIUM_MAX_LENGTH = None
FINBERT_LONG_MAX_LENGTH = None

CONFIG_ENV = {
    "SEC_CCM_DATA_PROFILE": DATA_PROFILE,
    "SEC_CCM_SAMPLE_ROOT": SAMPLE_ROOT,
    "SEC_CCM_WORK_ROOT": WORK_ROOT,
    "SEC_CCM_SEC_ZIP_DIR": SEC_ZIP_DIR,
    "SEC_CCM_SEC_BATCH_ROOT": SEC_BATCH_ROOT,
    "SEC_CCM_SEC_YEAR_MERGED_DIR": SEC_YEAR_MERGED_DIR,
    "SEC_CCM_SEC_LIGHT_METADATA_PATH": SEC_LIGHT_METADATA_PATH,
    "SEC_CCM_CCM_BASE_DIR": CCM_BASE_DIR,
    "SEC_CCM_CCM_DERIVED_DIR": CCM_DERIVED_DIR,
    "SEC_CCM_CCM_REUSE_DAILY_PATH": CCM_REUSE_DAILY_PATH,
    "SEC_CCM_CANONICAL_LINK_NAME": CANONICAL_LINK_NAME,
    "SEC_CCM_CCM_DAILY_NAME": CCM_DAILY_NAME,
    "SEC_CCM_RUN_ROOT": RUN_ROOT,
    "SEC_CCM_OUTPUT_DIR": SEC_CCM_OUTPUT_DIR,
    "SEC_CCM_ITEMS_ANALYSIS_DIR": SEC_ITEMS_ANALYSIS_DIR,
    "SEC_CCM_ITEMS_DIAGNOSTIC_DIR": SEC_ITEMS_DIAGNOSTIC_DIR,
    "SEC_CCM_NO_ITEM_DIR": SEC_NO_ITEM_DIR,
    "SEC_CCM_BOUNDARY_OUT_DIR": BOUNDARY_OUT_DIR,
    "SEC_CCM_BOUNDARY_INPUT_DIR": BOUNDARY_INPUT_DIR,
    "SEC_CCM_REFINITIV_STEP1_OUT_DIR": REFINITIV_STEP1_OUT_DIR,
    "SEC_CCM_REFINITIV_OWNERSHIP_UNIVERSE_DIR": REFINITIV_OWNERSHIP_UNIVERSE_DIR,
    "SEC_CCM_REFINITIV_OWNERSHIP_AUTHORITY_DIR": REFINITIV_OWNERSHIP_AUTHORITY_DIR,
    "SEC_CCM_REFINITIV_DOC_OWNERSHIP_LM2011_DIR": REFINITIV_DOC_OWNERSHIP_LM2011_DIR,
    "SEC_CCM_FF48_SICCODES_PATH": LM2011_FF48_SICCODES_PATH,
    "SEC_CCM_LOCAL_TMP": LOCAL_TMP,
    "SEC_CCM_LOCAL_WORK": LOCAL_WORK,
    "SEC_CCM_LOCAL_ITEM_WORK": LOCAL_ITEM_WORK,
    "SEC_CCM_LOCAL_MERGE_WORK": LOCAL_MERGE_WORK,
    "SEC_CCM_RUN_CCM_MODE": RUN_CCM_MODE,
    "SEC_CCM_RUN_SEC_PARSE": RUN_SEC_PARSE,
    "SEC_CCM_RUN_SEC_YEARLY_MERGE": RUN_SEC_YEARLY_MERGE,
    "SEC_CCM_RUN_SEC_CCM_PREMERGE": RUN_SEC_CCM_PREMERGE,
    "SEC_CCM_RUN_REFINITIV_STEP1": RUN_REFINITIV_STEP1,
    "SEC_CCM_RUN_REFINITIV_STEP1_RESOLUTION": RUN_REFINITIV_STEP1_RESOLUTION,
    "SEC_CCM_RUN_REFINITIV_OWNERSHIP_UNIVERSE_HANDOFF": RUN_REFINITIV_OWNERSHIP_UNIVERSE_HANDOFF,
    "SEC_CCM_RUN_REFINITIV_OWNERSHIP_UNIVERSE_RESULTS": RUN_REFINITIV_OWNERSHIP_UNIVERSE_RESULTS,
    "SEC_CCM_RUN_REFINITIV_OWNERSHIP_AUTHORITY": RUN_REFINITIV_OWNERSHIP_AUTHORITY,
    "SEC_CCM_RUN_REFINITIV_DOC_OWNERSHIP_LM2011_EXACT_HANDOFF": RUN_REFINITIV_DOC_OWNERSHIP_LM2011_EXACT_HANDOFF,
    "SEC_CCM_RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FALLBACK_HANDOFF": RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FALLBACK_HANDOFF,
    "SEC_CCM_RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FINALIZE": RUN_REFINITIV_DOC_OWNERSHIP_LM2011_FINALIZE,
    "SEC_CCM_RUN_REFINITIV_DOC_ANALYST_LM2011_ANCHORS": RUN_REFINITIV_DOC_ANALYST_LM2011_ANCHORS,
    "SEC_CCM_RUN_REFINITIV_DOC_ANALYST_LM2011_SELECT": RUN_REFINITIV_DOC_ANALYST_LM2011_SELECT,
    "SEC_CCM_RUN_REFINITIV_INSTRUMENT_AUTHORITY": RUN_REFINITIV_INSTRUMENT_AUTHORITY,
    "SEC_CCM_RUN_REFINITIV_ANALYST_REQUEST_GROUPS": RUN_REFINITIV_ANALYST_REQUEST_GROUPS,
    "SEC_CCM_RUN_REFINITIV_ANALYST_ACTUALS": RUN_REFINITIV_ANALYST_ACTUALS,
    "SEC_CCM_RUN_REFINITIV_ANALYST_ESTIMATES_MONTHLY": RUN_REFINITIV_ANALYST_ESTIMATES_MONTHLY,
    "SEC_CCM_RUN_REFINITIV_ANALYST_NORMALIZE": RUN_REFINITIV_ANALYST_NORMALIZE,
    "SEC_CCM_RUN_GATED_ITEM_EXTRACTION": RUN_GATED_ITEM_EXTRACTION,
    "SEC_CCM_RUN_UNMATCHED_DIAGNOSTIC_TRACK": RUN_UNMATCHED_DIAGNOSTIC_TRACK,
    "SEC_CCM_RUN_NO_ITEM_DIAGNOSTICS": RUN_NO_ITEM_DIAGNOSTICS,
    "SEC_CCM_RUN_BOUNDARY_DIAGNOSTICS": RUN_BOUNDARY_DIAGNOSTICS,
    "SEC_CCM_RUN_VALIDATION_CHECKS": RUN_VALIDATION_CHECKS,
    "SEC_CCM_RUN_LM2011_POST_REFINITIV": RUN_LM2011_POST_REFINITIV,
    "SEC_CCM_RUN_LM2011_SAMPLE_BACKBONE": RUN_LM2011_SAMPLE_BACKBONE,
    "SEC_CCM_RUN_LM2011_ANNUAL_ACCOUNTING_PANEL": RUN_LM2011_ANNUAL_ACCOUNTING_PANEL,
    "SEC_CCM_RUN_LM2011_QUARTERLY_ACCOUNTING_PANEL": RUN_LM2011_QUARTERLY_ACCOUNTING_PANEL,
    "SEC_CCM_RUN_LM2011_FF_FACTORS_DAILY_NORMALIZED": RUN_LM2011_FF_FACTORS_DAILY_NORMALIZED,
    "SEC_CCM_RUN_LM2011_FF_FACTORS_MONTHLY_WITH_MOM_NORMALIZED": RUN_LM2011_FF_FACTORS_MONTHLY_WITH_MOM_NORMALIZED,
    "SEC_CCM_RUN_LM2011_TEXT_FEATURES_FULL_10K": RUN_LM2011_TEXT_FEATURES_FULL_10K,
    "SEC_CCM_RUN_LM2011_TEXT_FEATURES_MDA": RUN_LM2011_TEXT_FEATURES_MDA,
    "SEC_CCM_RUN_LM2011_EVENT_SCREEN_SURFACE": RUN_LM2011_EVENT_SCREEN_SURFACE,
    "SEC_CCM_RUN_LM2011_TABLE_I_SAMPLE_CREATION": RUN_LM2011_TABLE_I_SAMPLE_CREATION,
    "SEC_CCM_RUN_LM2011_TABLE_I_SAMPLE_CREATION_1994_2024": RUN_LM2011_TABLE_I_SAMPLE_CREATION_1994_2024,
    "SEC_CCM_RUN_LM2011_EVENT_PANEL": RUN_LM2011_EVENT_PANEL,
    "SEC_CCM_RUN_LM2011_SUE_PANEL": RUN_LM2011_SUE_PANEL,
    "SEC_CCM_RUN_LM2011_RETURN_REGRESSION_PANEL_FULL_10K": RUN_LM2011_RETURN_REGRESSION_PANEL_FULL_10K,
    "SEC_CCM_RUN_LM2011_RETURN_REGRESSION_PANEL_MDA": RUN_LM2011_RETURN_REGRESSION_PANEL_MDA,
    "SEC_CCM_RUN_LM2011_SUE_REGRESSION_PANEL": RUN_LM2011_SUE_REGRESSION_PANEL,
    "SEC_CCM_RUN_LM2011_TABLE_IV_RESULTS": RUN_LM2011_TABLE_IV_RESULTS,
    "SEC_CCM_RUN_LM2011_TABLE_V_RESULTS": RUN_LM2011_TABLE_V_RESULTS,
    "SEC_CCM_RUN_LM2011_TABLE_VI_RESULTS": RUN_LM2011_TABLE_VI_RESULTS,
    "SEC_CCM_RUN_LM2011_TABLE_VIII_RESULTS": RUN_LM2011_TABLE_VIII_RESULTS,
    "SEC_CCM_RUN_LM2011_TABLE_IA_I_RESULTS": RUN_LM2011_TABLE_IA_I_RESULTS,
    "SEC_CCM_RUN_LM2011_TRADING_STRATEGY_MONTHLY_RETURNS": RUN_LM2011_TRADING_STRATEGY_MONTHLY_RETURNS,
    "SEC_CCM_RUN_LM2011_TABLE_IA_II_RESULTS": RUN_LM2011_TABLE_IA_II_RESULTS,
    "SEC_CCM_LM2011_OUTPUT_DIR": LM2011_OUTPUT_DIR,
    "SEC_CCM_LM2011_ADDITIONAL_DATA_DIR": LM2011_ADDITIONAL_DATA_DIR,
    "SEC_CCM_LM2011_SAMPLE_BACKBONE_PATH": LM2011_SAMPLE_BACKBONE_PATH,
    "SEC_CCM_LM2011_MATCHED_CLEAN_PATH": LM2011_MATCHED_CLEAN_PATH,
    "SEC_CCM_LM2011_DAILY_PANEL_PATH": LM2011_DAILY_PANEL_PATH,
    "SEC_CCM_LM2011_ITEMS_ANALYSIS_DIR": LM2011_ITEMS_ANALYSIS_DIR,
    "SEC_CCM_LM2011_CCM_BASE_DIR": LM2011_CCM_BASE_DIR,
    "SEC_CCM_LM2011_YEAR_MERGED_DIR": LM2011_YEAR_MERGED_DIR,
    "SEC_CCM_LM2011_MONTHLY_STOCK_PATH": LM2011_MONTHLY_STOCK_PATH,
    "SEC_CCM_LM2011_FF_MONTHLY_WITH_MOM_PATH": LM2011_FF_MONTHLY_WITH_MOM_PATH,
    "SEC_CCM_LM2011_FULL_10K_CLEANING_CONTRACT": LM2011_FULL_10K_CLEANING_CONTRACT,
    "SEC_CCM_LM2011_TEXT_FEATURE_BATCH_SIZE": LM2011_TEXT_FEATURE_BATCH_SIZE,
    "SEC_CCM_LM2011_EVENT_WINDOW_DOC_BATCH_SIZE": LM2011_EVENT_WINDOW_DOC_BATCH_SIZE,
    "SEC_CCM_RUN_FINBERT": RUN_FINBERT,
    "SEC_CCM_RUN_FINBERT_PREPROCESS": RUN_FINBERT_PREPROCESS,
    "SEC_CCM_RUN_FINBERT_ANALYSIS": RUN_FINBERT_ANALYSIS,
    "SEC_CCM_FINBERT_OUTPUT_DIR": FINBERT_OUTPUT_DIR,
    "SEC_CCM_FINBERT_SOURCE_ITEMS_DIR": FINBERT_SOURCE_ITEMS_DIR,
    "SEC_CCM_FINBERT_BACKBONE_PATH": FINBERT_BACKBONE_PATH,
    "SEC_CCM_FINBERT_YEARS": FINBERT_YEARS,
    "SEC_CCM_FINBERT_RUN_NAME": FINBERT_RUN_NAME,
    "SEC_CCM_FINBERT_DEVICE": FINBERT_DEVICE,
    "SEC_CCM_FINBERT_WRITE_SENTENCE_SCORES": FINBERT_WRITE_SENTENCE_SCORES,
    "SEC_CCM_FINBERT_SENTENCE_POSTPROCESS_POLICY": FINBERT_SENTENCE_POSTPROCESS_POLICY,
    "SEC_CCM_FINBERT_OVERWRITE": FINBERT_OVERWRITE,
    "SEC_CCM_FINBERT_NOTE": FINBERT_NOTE,
    "SEC_CCM_FINBERT_BATCH_PROFILE": FINBERT_BATCH_PROFILE,
    "SEC_CCM_FINBERT_SHORT_BATCH_SIZE": FINBERT_SHORT_BATCH_SIZE,
    "SEC_CCM_FINBERT_MEDIUM_BATCH_SIZE": FINBERT_MEDIUM_BATCH_SIZE,
    "SEC_CCM_FINBERT_LONG_BATCH_SIZE": FINBERT_LONG_BATCH_SIZE,
    "SEC_CCM_FINBERT_SHORT_MAX_LENGTH": FINBERT_SHORT_MAX_LENGTH,
    "SEC_CCM_FINBERT_MEDIUM_MAX_LENGTH": FINBERT_MEDIUM_MAX_LENGTH,
    "SEC_CCM_FINBERT_LONG_MAX_LENGTH": FINBERT_LONG_MAX_LENGTH,
    "SEC_CCM_SEC_PARSE_MODE": SEC_PARSE_MODE,
    "SEC_CCM_YEARS": YEARS,
    "SEC_CCM_ITEM_EXTRACTION_REGIME": ITEM_EXTRACTION_REGIME,
    "SEC_CCM_FORMS_10K_10Q": FORMS_10K_10Q,
    "SEC_CCM_DAILY_FEATURE_COLUMNS": DAILY_FEATURE_COLUMNS,
    "SEC_CCM_REQUIRED_DAILY_NON_NULL_FEATURES": REQUIRED_DAILY_NON_NULL_FEATURES,
    "SEC_CCM_LSEG_REQUEST_MIN_DATE": LSEG_REQUEST_MIN_DATE,
    "SEC_CCM_LSEG_REQUEST_MAX_DATE": LSEG_REQUEST_MAX_DATE,
}


def _encode_env_value(value: object) -> str:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, bool):
        return "true" if value else "false"
    if value is None:
        return "none"
    if isinstance(value, (list, tuple, dict)):
        return json.dumps(value)
    return str(value)


for key, value in CONFIG_ENV.items():
    os.environ[key] = _encode_env_value(value)

print(
    {
        "DATA_PROFILE": DATA_PROFILE,
        "WORK_ROOT": str(WORK_ROOT),
        "RUN_ROOT": str(RUN_ROOT),
        "SEC_ZIP_DIR": str(SEC_ZIP_DIR),
        "SEC_BATCH_ROOT": str(SEC_BATCH_ROOT),
        "CCM_BASE_DIR": str(CCM_BASE_DIR),
        "year_count": len(YEARS),
        "RUN_SEC_PARSE": RUN_SEC_PARSE,
        "RUN_SEC_YEARLY_MERGE": RUN_SEC_YEARLY_MERGE,
        "RUN_SEC_CCM_PREMERGE": RUN_SEC_CCM_PREMERGE,
        "RUN_LM2011_POST_REFINITIV": RUN_LM2011_POST_REFINITIV,
        "RUN_FINBERT": RUN_FINBERT,
        "RUN_FINBERT_PREPROCESS": RUN_FINBERT_PREPROCESS,
        "RUN_FINBERT_ANALYSIS": RUN_FINBERT_ANALYSIS,
        "FINBERT_SENTENCE_POSTPROCESS_POLICY": FINBERT_SENTENCE_POSTPROCESS_POLICY,
        "LM2011_OUTPUT_DIR": str(LM2011_OUTPUT_DIR),
        "FINBERT_OUTPUT_DIR": str(FINBERT_OUTPUT_DIR),
    }
)


In [ ]:
from thesis_pkg.notebooks_and_scripts.sec_ccm_unified_runner import main

main()
